# Custodian Column Semantic Mapper

This notebook builds, trains, evaluates, saves, and uses an ML model that maps unknown custodian file columns to canonical investment fields.

**Primary targets:** `quantity`, `price`, `value`

It also supports `currency`, `instrument_name`, `isin`, `trade_date`, `settlement_date`, `transaction_type`, and `other`.

The training data is synthetic and deliberately varies:
- custodian-specific terminology
- English, French, German, Spanish, Italian, Dutch, Portuguese and Afrikaans-like labels
- abbreviations and casing
- positions vs transactions
- numeric distributions
- column ordering
- decimal/thousands conventions

Evaluation holds out entire synthetic custodians so the model is tested against unseen source conventions.

In [ ]:
# Optional installation (uncomment if required)
# %pip install pandas numpy scikit-learn joblib matplotlib

In [ ]:
from __future__ import annotations

import math
import random
import re
import unicodedata
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ARTIFACT_DIR = Path("custodian_mapper_artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)

## 1. Canonical schema and multilingual header vocabulary

The model learns from header text **and** the data profile below each header. This is more robust than treating the problem as literal translation.

In [ ]:
CANONICAL_FIELDS = [
    "quantity", "price", "value", "currency", "instrument_name", "isin",
    "trade_date", "settlement_date", "transaction_type", "other"
]

HEADER_VARIANTS = {
    "quantity": [
        "quantity","qty","units","units held","holding quantity","shares","nominal",
        "nominal amount","par amount","position qty","balance",
        "quantité","qte","qté","nombre de titres",
        "menge","bestand","anzahl","stück","stueck",
        "cantidad","unidades","titulos","títulos",
        "quantità","unita","unità",
        "aantal","eenheden",
        "quantidade","unidades detidas",
        "hoeveelheid","eenhede"
    ],
    "price": [
        "price","unit price","market price","close price","trade price","px","rate",
        "cours","prix","prix unitaire",
        "kurs","preis","einstandskurs",
        "precio","precio unitario","cotización",
        "prezzo","prezzo unitario",
        "koers","prijs",
        "preço","preco","preço unitário",
        "prys","eenheidsprys"
    ],
    "value": [
        "value","market value","market value base","valuation","amount","gross amount",
        "position value","mv","notional value",
        "valeur","valeur de marché","montant",
        "marktwert","wert","betrag",
        "valor","valor de mercado","importe",
        "valore","valore di mercato","importo",
        "marktwaarde","waarde","bedrag",
        "valor de mercado","montante",
        "markwaarde","waarde"
    ],
    "currency": [
        "currency","ccy","curr","base currency","local currency",
        "devise","währung","waehrung","moneda","valuta","munt","moeda","geldeenheid"
    ],
    "instrument_name": [
        "instrument","security","security name","instrument name","asset","description",
        "libellé","titre","wertpapier","bezeichnung","instrumento","activo",
        "strumento","titolo","effect","omschrijving","ativo","bate"
    ],
    "isin": ["isin","isin code","security id","security identifier","id titre","wertpapierkennnummer"],
    "trade_date": [
        "trade date","transaction date","deal date","date de transaction",
        "handelsdatum","fecha operación","data operazione","transactiedatum","data negociação"
    ],
    "settlement_date": [
        "settlement date","settle date","value date","date de règlement",
        "valutadatum","fecha liquidación","data regolamento","afwikkeldatum"
    ],
    "transaction_type": [
        "transaction type","type","activity","event","action","type opération",
        "transaktionsart","tipo operación","tipo operazione","transactietype"
    ],
    "other": [
        "account","account number","portfolio","fund","custodian reference","reference",
        "book","strategy","country","exchange","sector","comment","status"
    ]
}

def mutate_header(s: str) -> str:
    """Introduce realistic source-system variations."""
    x = s
    if random.random() < 0.25:
        x = x.upper()
    elif random.random() < 0.25:
        x = x.title()
    if random.random() < 0.20:
        x = x.replace(" ", "_")
    if random.random() < 0.12:
        x = x.replace(" ", "")
    if random.random() < 0.12 and len(x) > 5:
        x = x[:max(3, int(len(x)*0.75))]
    return x

In [ ]:
INSTRUMENTS = [
    ("Apple Inc", "US0378331005", "USD"),
    ("Microsoft Corp", "US5949181045", "USD"),
    ("SAP SE", "DE0007164600", "EUR"),
    ("Nestle SA", "CH0038863350", "CHF"),
    ("Toyota Motor Corp", "JP3633400001", "JPY"),
    ("Example Global Equity Fund", "LU0000000001", "EUR"),
    ("Example Government Bond 2032", "ZA0000000002", "ZAR"),
]
TX_TYPES = ["BUY", "SELL", "DIVIDEND", "INTEREST", "TRANSFER", "SUBSCRIPTION", "REDEMPTION"]

def make_base_table(n=80, file_type="position"):
    rows = []
    for _ in range(n):
        name, isin, ccy = random.choice(INSTRUMENTS)
        qty = round(random.uniform(1, 10000), random.choice([0, 2, 4]))
        price = round(random.uniform(2, 2500), 4)
        value = round(qty * price, 2)

        trade_date = pd.Timestamp("2025-01-01") + pd.Timedelta(days=random.randint(0, 600))
        settle_date = trade_date + pd.Timedelta(days=random.choice([1,2,3]))

        rows.append({
            "instrument_name": name,
            "isin": isin,
            "quantity": qty,
            "price": price,
            "value": value,
            "currency": ccy,
            "trade_date": trade_date.strftime("%Y-%m-%d"),
            "settlement_date": settle_date.strftime("%Y-%m-%d"),
            "transaction_type": random.choice(TX_TYPES) if file_type == "transaction" else "POSITION",
            "other": f"REF-{random.randint(100000,999999)}"
        })
    return pd.DataFrame(rows)

In [ ]:
def parse_numeric_series(series: pd.Series) -> pd.Series:
    """Best-effort numeric conversion for profiling, including common locale formatting."""
    def conv(v):
        if pd.isna(v):
            return np.nan
        if isinstance(v, (int, float, np.number)):
            return float(v)
        s = str(v).strip()
        s = re.sub(r"[^\d,\.\-\(\)]", "", s)
        if not s:
            return np.nan
        neg = s.startswith("(") and s.endswith(")")
        s = s.strip("()")
        # Infer decimal separator from the last comma/dot.
        if "," in s and "." in s:
            if s.rfind(",") > s.rfind("."):
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", "")
        elif "," in s:
            tail = s.split(",")[-1]
            if len(tail) in (1, 2, 3, 4):
                s = s.replace(".", "").replace(",", ".")
            else:
                s = s.replace(",", "")
        try:
            out = float(s)
            return -out if neg else out
        except ValueError:
            return np.nan
    return series.map(conv)

def column_features(header: str, series: pd.Series, file_type: str) -> dict:
    s = series.dropna()
    numeric = parse_numeric_series(s)
    numeric_ratio = float(numeric.notna().mean()) if len(s) else 0.0
    vals = numeric.dropna()

    sample_strings = s.astype(str).head(30)
    date_ratio = float(pd.to_datetime(sample_strings, errors="coerce").notna().mean()) if len(sample_strings) else 0.0
    isin_ratio = float(sample_strings.str.match(r"^[A-Z]{2}[A-Z0-9]{9}[0-9]$").mean()) if len(sample_strings) else 0.0

    return {
        "header": str(header),
        "file_type": file_type,
        "numeric_ratio": numeric_ratio,
        "date_ratio": date_ratio,
        "isin_ratio": isin_ratio,
        "unique_ratio": float(s.nunique()/max(len(s),1)),
        "mean_abs": float(vals.abs().mean()) if len(vals) else 0.0,
        "median_abs": float(vals.abs().median()) if len(vals) else 0.0,
        "std": float(vals.std(ddof=0)) if len(vals) else 0.0,
        "integer_ratio": float(np.isclose(vals % 1, 0).mean()) if len(vals) else 0.0,
        "negative_ratio": float((vals < 0).mean()) if len(vals) else 0.0,
        "text_avg_len": float(sample_strings.str.len().mean()) if len(sample_strings) else 0.0,
    }

In [ ]:
def synthetic_training_examples(n_custodians=80, files_per_custodian=5, rows_per_file=60):
    records = []
    for custodian_id in range(n_custodians):
        # Each custodian receives a persistent preferred synonym for each semantic field.
        preferred = {
            field: random.choice(variants)
            for field, variants in HEADER_VARIANTS.items()
        }

        for file_no in range(files_per_custodian):
            file_type = random.choice(["position", "transaction"])
            base = make_base_table(rows_per_file, file_type)

            for field in CANONICAL_FIELDS:
                # Mostly persistent convention, sometimes an alternative export label.
                header = preferred[field] if random.random() < 0.75 else random.choice(HEADER_VARIANTS[field])
                header = mutate_header(header)
                feat = column_features(header, base[field], file_type)
                feat.update({
                    "label": field,
                    "custodian_id": f"CUST_{custodian_id:03d}",
                    "file_no": file_no,
                })
                records.append(feat)

    return pd.DataFrame(records)

examples = synthetic_training_examples()
print(examples.shape)
examples.head()

## 2. Hold out entire custodians

This avoids an overly optimistic random row split. The test set represents custodians whose preferred naming conventions were not used for fitting.

In [ ]:
custodians = sorted(examples["custodian_id"].unique())
rng = np.random.default_rng(SEED)
rng.shuffle(custodians)

cut = int(len(custodians) * 0.80)
train_custodians = set(custodians[:cut])
test_custodians = set(custodians[cut:])

train_df = examples[examples.custodian_id.isin(train_custodians)].copy()
test_df = examples[examples.custodian_id.isin(test_custodians)].copy()

print("Training custodians:", len(train_custodians))
print("Held-out custodians:", len(test_custodians))
print("Training examples:", len(train_df))
print("Test examples:", len(test_df))

## 3. Train the classifier

Character n-grams are useful for multilingual words, abbreviations, spelling variants and truncated headers. Numeric/data-profile features help distinguish fields with similar or unknown headers.

In [ ]:
NUMERIC_FEATURES = [
    "numeric_ratio","date_ratio","isin_ratio","unique_ratio",
    "mean_abs","median_abs","std","integer_ratio","negative_ratio","text_avg_len"
]

# We combine header and file context into one text field.
for frame in (train_df, test_df):
    frame["text"] = frame["header"].astype(str) + " __FILE_" + frame["file_type"].astype(str)

preprocessor = ColumnTransformer(
    transformers=[
        ("header_char", TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), min_df=1), "text"),
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
    ]
)

# Logistic regression handles sparse TF-IDF particularly well for this problem.
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("features", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2500,
        class_weight="balanced",
        C=4.0,
        random_state=SEED
    ))
])

model.fit(train_df[["text"] + NUMERIC_FEATURES], train_df["label"])
print("Model trained.")

In [ ]:
pred = model.predict(test_df[["text"] + NUMERIC_FEATURES])
accuracy = accuracy_score(test_df["label"], pred)

print(f"Held-out custodian accuracy: {accuracy:.4%}\n")
print(classification_report(test_df["label"], pred, digits=4))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_predictions(
    test_df["label"], pred,
    xticks_rotation=45,
    normalize="true",
    ax=ax
)
ax.set_title("Normalized confusion matrix — held-out custodians")
plt.tight_layout()
plt.show()

## 4. Save the trained model

In [ ]:
MODEL_PATH = ARTIFACT_DIR / "custodian_column_mapper.joblib"
joblib.dump(model, MODEL_PATH)
print("Saved:", MODEL_PATH.resolve())

## 5. Inference

`predict_columns()` accepts an arbitrary positions/transactions DataFrame and returns the predicted canonical field plus confidence and alternatives.

The confidence threshold is deliberately configurable. In production, low-confidence mappings should be sent for human review rather than silently accepted.

In [ ]:
def predict_columns(
    df: pd.DataFrame,
    trained_model=model,
    file_type: str = "position",
    confidence_threshold: float = 0.70,
) -> pd.DataFrame:
    feature_rows = []
    for col in df.columns:
        f = column_features(str(col), df[col], file_type)
        f["text"] = f["header"] + " __FILE_" + file_type
        feature_rows.append(f)

    features = pd.DataFrame(feature_rows)
    X = features[["text"] + NUMERIC_FEATURES]

    probabilities = trained_model.predict_proba(X)
    classes = trained_model.classes_

    output = []
    for i, source_col in enumerate(df.columns):
        order = np.argsort(probabilities[i])[::-1]
        best = order[0]
        second = order[1]

        confidence = float(probabilities[i, best])
        output.append({
            "source_column": source_col,
            "predicted_field": classes[best],
            "confidence": round(confidence, 4),
            "review_required": confidence < confidence_threshold,
            "second_choice": classes[second],
            "second_confidence": round(float(probabilities[i, second]), 4),
        })

    return pd.DataFrame(output).sort_values("confidence", ascending=False)

In [ ]:
# Example: unseen German-style positions export
incoming = pd.DataFrame({
    "Wertpapier": ["Asset A", "Asset B", "Asset C", "Asset D"],
    "Bestand": [1000, 500, 250, 1200],
    "Kurs": [23.45, 81.10, 102.25, 12.50],
    "Marktwert": [23450.00, 40550.00, 25562.50, 15000.00],
    "Währung": ["EUR", "EUR", "CHF", "EUR"],
    "ISIN Code": ["DE0000000011","DE0000000029","CH0000000037","DE0000000045"],
})

mapping = predict_columns(incoming, file_type="position")
mapping

## 6. Financial relationship validation

Header classification can be strengthened with domain rules. For positions, the most useful relationship is often:

`quantity × price ≈ value`

The function below scores candidate triplets independently of the language used in their headers.

In [ ]:
def relationship_error(df, quantity_col, price_col, value_col):
    q = parse_numeric_series(df[quantity_col])
    p = parse_numeric_series(df[price_col])
    v = parse_numeric_series(df[value_col])

    mask = q.notna() & p.notna() & v.notna() & (v.abs() > 1e-12)
    if mask.sum() == 0:
        return np.inf

    relative_error = ((q[mask] * p[mask] - v[mask]).abs() / v[mask].abs())
    return float(relative_error.median())

def find_quantity_price_value_relationship(df):
    numeric_cols = [
        c for c in df.columns
        if parse_numeric_series(df[c]).notna().mean() >= 0.70
    ]

    candidates = []
    for q in numeric_cols:
        for p in numeric_cols:
            if p == q:
                continue
            for v in numeric_cols:
                if v in (q, p):
                    continue
                err = relationship_error(df, q, p, v)
                if np.isfinite(err):
                    candidates.append((err, q, p, v))

    candidates.sort(key=lambda x: x[0])
    return pd.DataFrame(
        candidates[:10],
        columns=["median_relative_error", "quantity_candidate", "price_candidate", "value_candidate"]
    )

find_quantity_price_value_relationship(incoming)

## 7. Combined inference for quantity / price / value

The next function combines the ML predictions with the arithmetic relationship. When a strong `q × p ≈ v` triplet exists, it can resolve unfamiliar headers that the language model has not previously encountered.

In [ ]:
def infer_custodian_schema(
    df: pd.DataFrame,
    trained_model=model,
    file_type="position",
    confidence_threshold=0.70,
    relationship_tolerance=0.02,
):
    predictions = predict_columns(
        df, trained_model, file_type, confidence_threshold
    ).reset_index(drop=True)

    relationships = find_quantity_price_value_relationship(df)
    relationship_used = None

    if len(relationships):
        best = relationships.iloc[0]
        if best["median_relative_error"] <= relationship_tolerance:
            relationship_used = {
                "quantity": best["quantity_candidate"],
                "price": best["price_candidate"],
                "value": best["value_candidate"],
                "median_relative_error": float(best["median_relative_error"]),
            }

            # Domain evidence is very strong, but retain transparency that it was rule-assisted.
            for semantic in ("quantity", "price", "value"):
                source = relationship_used[semantic]
                mask = predictions["source_column"] == source
                predictions.loc[mask, "predicted_field"] = semantic
                predictions.loc[mask, "review_required"] = False
                predictions.loc[mask, "domain_rule"] = "quantity × price ≈ value"

    return predictions, relationship_used

final_mapping, relationship = infer_custodian_schema(incoming)
print("Detected relationship:", relationship)
final_mapping

## 8. Load the saved model and infer on a real custodian CSV

Replace the filename and file type below when you receive a sample from an authorized custodian.

Do not automatically ingest low-confidence mappings into downstream accounting/investment systems. Persist the source header, prediction, confidence, model version and any human override for auditability.

In [ ]:
# loaded_model = joblib.load("custodian_mapper_artifacts/custodian_column_mapper.joblib")
# custodian_df = pd.read_csv("sample_custodian_positions.csv")
#
# mapping, relationship = infer_custodian_schema(
#     custodian_df,
#     trained_model=loaded_model,
#     file_type="position",   # or "transaction"
#     confidence_threshold=0.80,
# )
#
# display(mapping)
# print(relationship)

## Recommended next iteration

Synthetic data is useful for bootstrapping, but the production model should progressively learn from **approved mappings from real custodians**.

A practical production loop is:

1. Predict mappings.
2. Automatically accept only high-confidence, validated mappings.
3. Send ambiguous columns to review.
4. Store the reviewer-approved source → canonical mapping.
5. Add approved examples to the training corpus.
6. Retrain/version the model.
7. Evaluate against custodians excluded from training.
8. Promote only models that meet defined acceptance thresholds.

For investment data, also consider separate semantic rules for bonds, FX, derivatives, accrued interest and transaction cash amounts because `quantity × price = value` does not universally apply without multipliers, FX conversion or instrument-specific conventions.